# Customer Retention Intelligence — API Integration Testing (Phase 11)

## Objective

Exercise the Phase 11 FastAPI service end-to-end:

```
API Request -> Pydantic validation -> feature preparation
    -> MLflow registered model (Phase 10) -> churn probability
    -> risk level -> SHAP explanation (Phase 8) -> next-best action (Phase 9)
    -> JSON response
```

**This notebook does not retrain, retune, or modify the Phase 7 model.**
It calls the running API, which itself only loads the already-registered
MLflow model (Phase 10) and reuses Phase 8's SHAP logic and Phase 9's
recommendation rules unmodified (see `api/services.py`).

This notebook uses FastAPI's `TestClient` rather than a separately
launched `uvicorn` process, so it can run non-interactively end-to-end
without needing a background server -- `TestClient` drives the exact same
ASGI application object (`api.main.app`), including its startup/shutdown
lifecycle, so this is a faithful integration test, not a mock.

To run the real HTTP server instead (e.g. for manual/browser testing):

```bash
uvicorn api.main:app --reload
```

then visit `http://127.0.0.1:8000/docs`.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import JSON, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

from fastapi.testclient import TestClient

from src.models import baseline as bl
from api.main import app
from api.services import feature_row_to_payload

PROJECT_ROOT

/Users/dhananjaymandal/customer-retention-intelligence/.venv/lib/python3.14/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


PosixPath('/Users/dhananjaymandal/customer-retention-intelligence')

## 1. Start the API (via TestClient)

`TestClient` used as a context manager runs the app's full `lifespan`
startup -- the same code path `uvicorn api.main:app` runs -- so the
registered Phase 10 model, SHAP context, and recommendation config are
all loaded exactly once here, for real.

In [2]:
client = TestClient(app)
client.__enter__()  # keep the app's lifespan (and loaded model) alive for the rest of this notebook
print("API started.")

2026-09-23 12:04:53,779 INFO api.services: Model service ready: model=customer-churn-logistic-regression version=1 features=131


API started.


## 2. Call GET /health

In [3]:
response = client.get("/health")
print(f"status_code: {response.status_code}")
display(JSON(response.json()))

2026-09-23 12:04:53,789 INFO httpx: HTTP Request: GET http://testserver/health "HTTP/1.1 200 OK"


status_code: 200


<IPython.core.display.JSON object>

## 3. Call GET /model-info

Safe, whitelisted metadata only -- confirms the exact Phase 7
hyperparameters (`C=0.01`, `class_weight="balanced"`) are what's being
served, with no MLflow internals or filesystem paths exposed.

In [4]:
response = client.get("/model-info")
print(f"status_code: {response.status_code}")
display(JSON(response.json()))

2026-09-23 12:04:53,794 INFO httpx: HTTP Request: GET http://testserver/model-info "HTTP/1.1 200 OK"


status_code: 200


<IPython.core.display.JSON object>

## 4. Build a Request From a Real Feature Row

Using a real row from `data/processed/customer_features.parquet` (via
`api.services.feature_row_to_payload`) rather than inventing feature
values -- exactly as every other phase's notebooks have done.

In [5]:
features = bl.load_feature_dataset()
sample_row = features.sample(n=1, random_state=123).iloc[0]
sample_payload = feature_row_to_payload(sample_row)

print(f"customer_id: {sample_payload['customer_id']}")
print(f"snapshot_date: {sample_payload['snapshot_date']}")
print(f"Total fields in payload: {len(sample_payload)}")

customer_id: C002081
snapshot_date: 2026-03-31
Total fields in payload: 133


## 5. Send a Valid Prediction Request (POST /predict)

In [6]:
response = client.post("/predict", json=sample_payload)
print(f"status_code: {response.status_code}")
prediction = response.json()
display(JSON(prediction))

2026-09-23 12:04:53,853 INFO api.main: POST /predict customer_id=C002081


2026-09-23 12:04:53,858 INFO httpx: HTTP Request: POST http://testserver/predict "HTTP/1.1 200 OK"


status_code: 200


<IPython.core.display.JSON object>

## 6. Request a SHAP Explanation (POST /predict/explain)

Reuses Phase 8's `explain_customer` unmodified -- every driver here comes
from the model's actual SHAP values for this specific customer, and is
labeled a `model_signal`, never a causal claim (see the `disclaimer`
field).

In [7]:
response = client.post("/predict/explain", json={**sample_payload, "top_k": 5})
print(f"status_code: {response.status_code}")
explanation = response.json()

print(f"churn_probability: {explanation['churn_probability']:.4f}")
print(f"risk_level: {explanation['risk_level']}")
print("\nTop positive (risk-raising) drivers:")
for driver in explanation["top_positive_drivers"]:
    print(f"  {driver['feature']:35s} shap={driver['shap_value']:+.4f}  customer_value={driver['customer_value']}")
print("\nTop negative (risk-lowering) drivers:")
for driver in explanation["top_negative_drivers"]:
    print(f"  {driver['feature']:35s} shap={driver['shap_value']:+.4f}  customer_value={driver['customer_value']}")
print(f"\nDisclaimer: {explanation['disclaimer']}")

2026-09-23 12:04:53,863 INFO api.main: POST /predict/explain customer_id=C002081


2026-09-23 12:04:53,874 INFO httpx: HTTP Request: POST http://testserver/predict/explain "HTTP/1.1 200 OK"


status_code: 200
churn_probability: 0.8305
risk_level: critical

Top positive (risk-raising) drivers:
  active_days_90d                     shap=+0.9329  customer_value=35.0
  avg_pages_viewed_90d                shap=+0.5450  customer_value=3.6176470588235294
  avg_active_minutes_previous30d      shap=+0.3210  customer_value=4.959999999999999
  avg_session_minutes_previous30d     shap=+0.3106  customer_value=6.202727272727273
  resolved_ticket_count_180d          shap=+0.2982  customer_value=7.0

Top negative (risk-lowering) drivers:
  avg_active_minutes_90d              shap=-0.3429  customer_value=6.482647058823529
  avg_feature_usage_count_previous30d shap=-0.3271  customer_value=0.9
  resolved_ticket_count_90d           shap=-0.3181  customer_value=4.0
  avg_pages_viewed_recent30d          shap=-0.2605  customer_value=3.3846153846153846
  category_feature_request_count_30d  shap=-0.2477  customer_value=1.0

Disclaimer: Each driver is a model_signal: an association this model's SHAP

## 7. Request a Next-Best-Action Recommendation (POST /recommend)

Reuses Phase 9's rule engine unmodified.

In [8]:
response = client.post("/recommend", json=sample_payload)
print(f"status_code: {response.status_code}")
recommendation = response.json()
display(JSON(recommendation))

2026-09-23 12:04:53,879 INFO api.main: POST /recommend customer_id=C002081


2026-09-23 12:04:53,886 INFO httpx: HTTP Request: POST http://testserver/recommend "HTTP/1.1 200 OK"


status_code: 200


<IPython.core.display.JSON object>

## 8. Request the Combined Customer Analysis (POST /customer-analysis)

Built for the Streamlit dashboard: one response bundling prediction,
explanation, and recommendation, guaranteed internally consistent because
`churn_probability` is computed once and shared across all three
sections.

In [9]:
response = client.post("/customer-analysis", json={**sample_payload, "top_k": 3})
print(f"status_code: {response.status_code}")
analysis = response.json()

print("Sections returned:", list(analysis.keys()))
print(f"\nprediction.churn_probability     = {analysis['prediction']['churn_probability']:.4f}")
print(f"explanation.churn_probability    = {analysis['explanation']['churn_probability']:.4f}")
print(f"recommendation.churn_probability = {analysis['recommendation']['churn_probability']:.4f}")
assert (
    analysis["prediction"]["churn_probability"]
    == analysis["explanation"]["churn_probability"]
    == analysis["recommendation"]["churn_probability"]
)
print("\nAll three sections agree on churn_probability -- internally consistent.")

2026-09-23 12:04:53,891 INFO api.main: POST /customer-analysis customer_id=C002081


2026-09-23 12:04:53,908 INFO httpx: HTTP Request: POST http://testserver/customer-analysis "HTTP/1.1 200 OK"


status_code: 200
Sections returned: ['customer', 'prediction', 'explanation', 'recommendation']

prediction.churn_probability     = 0.8305
explanation.churn_probability    = 0.8305
recommendation.churn_probability = 0.8305

All three sections agree on churn_probability -- internally consistent.


In [10]:
display(JSON(analysis))

<IPython.core.display.JSON object>

## 9. Demonstrate a Validation Failure

Two realistic ways a request can be rejected: an entirely missing
required feature, and an out-of-range/invalid value. Both return **HTTP
422**, with a clear, structured error body -- no stack trace, no internal
file paths.

In [11]:
incomplete_payload = dict(sample_payload)
del incomplete_payload["age"]  # a required feature, entirely omitted

response = client.post("/predict", json=incomplete_payload)
print(f"Missing required feature -> status_code: {response.status_code}")
display(JSON(response.json()))

2026-09-23 12:04:53,915 INFO httpx: HTTP Request: POST http://testserver/predict "HTTP/1.1 422 Unprocessable Entity"


Missing required feature -> status_code: 422


<IPython.core.display.JSON object>

In [12]:
invalid_payload = dict(sample_payload)
invalid_payload["age"] = "not-a-number"  # wrong type entirely

response = client.post("/predict", json=invalid_payload)
print(f"Invalid numeric value -> status_code: {response.status_code}")
display(JSON(response.json()))

2026-09-23 12:04:53,920 INFO httpx: HTTP Request: POST http://testserver/predict "HTTP/1.1 422 Unprocessable Entity"


Invalid numeric value -> status_code: 422


<IPython.core.display.JSON object>

In [13]:
malformed_date_payload = dict(sample_payload)
malformed_date_payload["snapshot_date"] = "not-a-real-date"

response = client.post("/predict", json=malformed_date_payload)
print(f"Malformed date -> status_code: {response.status_code}")
display(JSON(response.json()))

2026-09-23 12:04:53,924 INFO httpx: HTTP Request: POST http://testserver/predict "HTTP/1.1 422 Unprocessable Entity"


Malformed date -> status_code: 422


<IPython.core.display.JSON object>

## 10. API Documentation

FastAPI auto-generates interactive docs from the schemas and endpoint
descriptions defined in `api/main.py` / `api/schemas.py`.

In [14]:
for path in ("/docs", "/redoc", "/openapi.json"):
    response = client.get(path)
    print(f"{path}: {response.status_code}")

2026-09-23 12:04:53,928 INFO httpx: HTTP Request: GET http://testserver/docs "HTTP/1.1 200 OK"


2026-09-23 12:04:53,929 INFO httpx: HTTP Request: GET http://testserver/redoc "HTTP/1.1 200 OK"


/docs: 200
/redoc: 200


2026-09-23 12:04:54,079 INFO httpx: HTTP Request: GET http://testserver/openapi.json "HTTP/1.1 200 OK"


/openapi.json: 200


## 11. Shut Down the TestClient

In [15]:
client.__exit__(None, None, None)
print("API stopped.")

API stopped.


## Summary

- Every endpoint (`/health`, `/model-info`, `/predict`, `/predict/explain`, `/recommend`, `/customer-analysis`) responded successfully against the **real registered Phase 10 model**.
- `/predict/explain` used **Phase 8's SHAP logic unmodified**; `/recommend` used **Phase 9's rule engine unmodified**.
- `/customer-analysis`'s three sections were confirmed **internally consistent** (identical `churn_probability`).
- Invalid requests (missing feature, wrong type, malformed date) were all rejected with **HTTP 422**, never a stack trace.
- **No model was retrained, tuned, or modified anywhere in this notebook.**